# Two-Wallet Framework: Build a Commodity Allocator

**Goal:** Build a complete two-wallet commodity allocation system with Thompson Sampling in 15 minutes.

**What you'll learn:**
- Implement core wallet (stable, equal-weight) + bandit sleeve (adaptive)
- Backtest on real commodity futures data
- Visualize how the bandit learns and adapts
- Compare to baseline equal-weight strategy

**Data:** Real commodity futures from Yahoo Finance (WTI, Gold, Copper, NatGas, Corn)

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries loaded successfully!")

In [ ]:
learning_objectives(["The two-wallet framework separates stable core from adaptive bandit", "Thompson Sampling learns optimal allocation from historical performance", "Guardrails (min/max allocation) prevent concentration risk", "Backtesting shows how the system adapts over time"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Step 1: Load Real Commodity Data

We'll use 3 years of historical data for 5 commodity futures:
- **CL=F**: WTI Crude Oil
- **GC=F**: Gold
- **HG=F**: Copper  
- **NG=F**: Natural Gas
- **ZC=F**: Corn

In [ ]:
# Define commodity tickers
commodities = {
    'WTI': 'CL=F',
    'Gold': 'GC=F',
    'Copper': 'HG=F',
    'NatGas': 'NG=F',
    'Corn': 'ZC=F'
}

# Download data with error handling
try:
    data = yf.download(
        list(commodities.values()),
        start='2021-01-01',
        end='2024-01-01',
        progress=False
    )['Adj Close']
    
    # Rename columns to commodity names
    data.columns = list(commodities.keys())
    
    print(f"Downloaded {len(data)} days of data")
    print(f"Date range: {data.index[0]} to {data.index[-1]}")
    
except Exception as e:
    print(f"Warning: Could not download data ({e})")
    print("Using synthetic data for demonstration...\n")
    
    # Synthetic fallback data
    dates = pd.date_range('2021-01-01', '2024-01-01', freq='D')
    np.random.seed(42)
    data = pd.DataFrame({
        'WTI': 70 + np.cumsum(np.random.randn(len(dates)) * 2),
        'Gold': 1800 + np.cumsum(np.random.randn(len(dates)) * 10),
        'Copper': 4.0 + np.cumsum(np.random.randn(len(dates)) * 0.05),
        'NatGas': 3.0 + np.cumsum(np.random.randn(len(dates)) * 0.3),
        'Corn': 550 + np.cumsum(np.random.randn(len(dates)) * 5)
    }, index=dates)

# Compute weekly returns
weekly_prices = data.resample('W').last()
weekly_returns = weekly_prices.pct_change().dropna()

print(f"\nWeekly returns shape: {weekly_returns.shape}")
weekly_returns.head()

## Step 2: Thompson Sampling Bandit

The core algorithm that learns which commodities perform well.

In [ ]:
class ThompsonSamplingBandit:
    """
    Thompson Sampling with Gaussian rewards.
    """
    def __init__(self, K, prior_mean=0.001, prior_std=0.02):
        self.K = K
        self.means = np.full(K, prior_mean)
        self.stds = np.full(K, prior_std)
        self.n = np.zeros(K)
    
    def get_allocation(self):
        """Get allocation weights using Thompson Sampling."""
        # Sample from each arm's posterior
        samples = np.random.normal(self.means, self.stds)
        
        # Convert to weights via softmax
        exp_samples = np.exp(samples - samples.max())
        weights = exp_samples / exp_samples.sum()
        
        return weights
    
    def update(self, returns):
        """Update beliefs based on observed returns."""
        for i in range(self.K):
            self.n[i] += 1
            lr = 1 / (self.n[i] + 1)
            self.means[i] = (1 - lr) * self.means[i] + lr * returns[i]
            self.stds[i] = self.stds[i] / np.sqrt(1 + self.n[i])

# Test it
bandit = ThompsonSamplingBandit(K=5)
print("Initial allocation:", bandit.get_allocation())
print("After one update:")
bandit.update(np.array([0.02, -0.01, 0.01, 0.03, -0.01]))
print(bandit.get_allocation())

## Step 3: Two-Wallet System

Combine stable core (80%) with adaptive bandit sleeve (20%).

In [ ]:
class TwoWalletBandit:
    """
    Two-wallet framework: Core + Bandit sleeve.
    """
    def __init__(
        self, 
        K, 
        core_pct=0.8, 
        bandit_pct=0.2,
        min_alloc=0.05,
        max_alloc=0.40
    ):
        self.K = K
        self.core_pct = core_pct
        self.bandit_pct = bandit_pct
        self.min_alloc = min_alloc
        self.max_alloc = max_alloc
        self.bandit = ThompsonSamplingBandit(K)
    
    def get_weights(self):
        """Get combined allocation weights."""
        # Core: equal-weight
        core_weights = np.ones(self.K) / self.K
        
        # Bandit: Thompson Sampling
        bandit_weights = self.bandit.get_allocation()
        
        # Apply guardrails to bandit
        bandit_weights = np.clip(bandit_weights, self.min_alloc, self.max_alloc)
        bandit_weights = bandit_weights / bandit_weights.sum()
        
        # Combine
        total = self.core_pct * core_weights + self.bandit_pct * bandit_weights
        return total, core_weights, bandit_weights
    
    def update(self, returns):
        """Update bandit beliefs."""
        self.bandit.update(returns)

# Test it
two_wallet = TwoWalletBandit(K=5)
total, core, bandit = two_wallet.get_weights()
print("Core allocation:", core)
print("Bandit allocation:", bandit)
print("Total allocation:", total)

## Step 4: Backtest the System

Run the two-wallet bandit on historical data and compare to equal-weight baseline.

In [ ]:
def backtest_two_wallet(returns_df, core_pct=0.8, bandit_pct=0.2):
    """
    Backtest two-wallet bandit on historical returns.
    """
    K = len(returns_df.columns)
    bandit = TwoWalletBandit(K, core_pct, bandit_pct)
    
    results = []
    
    for date, row in returns_df.iterrows():
        # Get current allocation
        total_weights, core_weights, bandit_weights = bandit.get_weights()
        
        # Compute portfolio return
        returns_array = row.values
        portfolio_return = np.dot(total_weights, returns_array)
        equal_weight_return = np.mean(returns_array)
        
        # Store results
        results.append({
            'date': date,
            'two_wallet_return': portfolio_return,
            'equal_weight_return': equal_weight_return,
            **{f'weight_{col}': w for col, w in zip(returns_df.columns, total_weights)}
        })
        
        # Update bandit
        bandit.update(returns_array)
    
    return pd.DataFrame(results).set_index('date')

# Run backtest
print("Running backtest...")
backtest_results = backtest_two_wallet(weekly_returns)

# Compute cumulative returns
backtest_results['two_wallet_cumulative'] = (
    (1 + backtest_results['two_wallet_return']).cumprod()
)
backtest_results['equal_weight_cumulative'] = (
    (1 + backtest_results['equal_weight_return']).cumprod()
)

print("\nBacktest complete!")
print(f"Weeks: {len(backtest_results)}")
print(f"Two-Wallet Total Return: {backtest_results['two_wallet_cumulative'].iloc[-1] - 1:.2%}")
print(f"Equal-Weight Total Return: {backtest_results['equal_weight_cumulative'].iloc[-1] - 1:.2%}")

## Step 5: Visualize Results

See how the two-wallet system performed vs equal-weight baseline.

In [ ]:
# Plot cumulative returns
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    backtest_results.index,
    backtest_results['two_wallet_cumulative'],
    label='Two-Wallet Bandit (80% Core + 20% Adaptive)',
    linewidth=2,
    color='#2E86AB'
)
ax.plot(
    backtest_results.index,
    backtest_results['equal_weight_cumulative'],
    label='Equal-Weight Baseline',
    linewidth=2,
    linestyle='--',
    color='#A23B72'
)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return (Base = 1.0)', fontsize=12)
ax.set_title('Two-Wallet Bandit vs Equal-Weight Baseline', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight: The two-wallet system adapts allocation over time,")
print("tilting toward better performers while maintaining core stability.")

## Step 6: Allocation Tilts Over Time

Visualize how the bandit adapted allocation across commodities.

In [ ]:
# Extract allocation weights
weight_cols = [col for col in backtest_results.columns if col.startswith('weight_')]
weights_df = backtest_results[weight_cols]
weights_df.columns = [col.replace('weight_', '') for col in weights_df.columns]

# Plot stacked area chart
fig, ax = plt.subplots(figsize=(12, 6))

ax.stackplot(
    weights_df.index,
    *[weights_df[col] for col in weights_df.columns],
    labels=weights_df.columns,
    alpha=0.8
)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Allocation Weight', fontsize=12)
ax.set_title('Two-Wallet Allocation Over Time', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNotice how allocations shift gradually as the bandit learns.")
print("The 80% core keeps it stable; the 20% bandit provides adaptive tilts.")

## Step 7: Performance Metrics

Compare key metrics between strategies.

In [ ]:
def compute_metrics(returns):
    """Compute performance metrics."""
    total_return = (1 + returns).prod() - 1
    annual_return = (1 + total_return) ** (52 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(52)
    sharpe = annual_return / volatility if volatility > 0 else 0
    
    cumulative = (1 + returns).cumprod()
    drawdown = (cumulative / cumulative.cummax() - 1).min()
    
    return {
        'Total Return': f"{total_return:.2%}",
        'Annual Return': f"{annual_return:.2%}",
        'Volatility': f"{volatility:.2%}",
        'Sharpe Ratio': f"{sharpe:.2f}",
        'Max Drawdown': f"{drawdown:.2%}"
    }

# Compute metrics
two_wallet_metrics = compute_metrics(backtest_results['two_wallet_return'])
equal_weight_metrics = compute_metrics(backtest_results['equal_weight_return'])

# Display
comparison = pd.DataFrame({
    'Two-Wallet Bandit': two_wallet_metrics,
    'Equal-Weight': equal_weight_metrics
})

print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(comparison)
print("="*60)

## Modify This: Experiment with Parameters

Try changing these parameters and rerun the backtest:

1. **Core/Bandit split**: Change `core_pct` and `bandit_pct`
   - Try 70/30 (more aggressive)
   - Try 90/10 (more conservative)

2. **Guardrails**: Adjust `min_alloc` and `max_alloc`
   - Tighter: `min_alloc=0.10, max_alloc=0.30`
   - Looser: `min_alloc=0.02, max_alloc=0.60`

3. **Different commodities**: Add more arms
   - Include Silver (SI=F), Platinum (PL=F)
   - Include Soybeans (ZS=F), Wheat (ZW=F)

**Challenge:** What parameter settings give the best Sharpe ratio?

In [ ]:
# Experiment here
# Modify core_pct, bandit_pct, min_alloc, max_alloc

custom_backtest = backtest_two_wallet(
    weekly_returns,
    core_pct=0.70,  # <-- Try changing this
    bandit_pct=0.30  # <-- And this
)

custom_metrics = compute_metrics(custom_backtest['two_wallet_return'])
print("\nCustom Configuration Metrics:")
for metric, value in custom_metrics.items():
    print(f"  {metric}: {value}")

## Summary

You just built a complete two-wallet commodity allocation system!

**What you learned:**
1. The two-wallet framework separates stable core from adaptive bandit
2. Thompson Sampling learns optimal allocation from historical performance
3. Guardrails (min/max allocation) prevent concentration risk
4. Backtesting shows how the system adapts over time

**What's next:**
- [Reward Function Lab](02_reward_function_lab.ipynb): See how different rewards change behavior
- [Regime-Aware Bandit](03_regime_commodity_bandit.ipynb): Add market regime detection
- [Guardrails Guide](../guides/03_guardrails_and_safety.md): Add more safety constraints

**Real-world deployment:**
- Add transaction cost modeling
- Include slippage and execution lag
- Connect to live futures data API
- Add monitoring and alerts

In [ ]:
key_takeaways(["1. The two-wallet framework separates stable core from adaptive bandit", "- [Reward Function Lab](02_reward_function_lab.ipynb): See how different rewards", "[Regime-Aware Bandit](03_regime_commodity_bandit.ipynb): Add market regime detec", "[Guardrails Guide](../guides/03_guardrails_and_safety.md): Add more safety const", "- Add transaction cost modeling", "Include slippage and execution lag"])